# Đo Deformable 3DGS, SpectroMotion, RT-Splatting và VIGOR-GS trên vùng kính (transparent mask)

Notebook này chấm lại ảnh render test **đã có sẵn trên Drive** của từng method. Không train lại gì. Mọi method được so trên **cùng dữ liệu hiện tại đã căn chỉnh** của scene (`colmap/images`, `colmap/masks`, split test trong `colmap/split.json`) và ở **cùng độ phân giải mà VIGOR-GS được chấm**. Phần tính toán nằm trong `eval_masked_baselines.py`; docstring của file đó mô tả chi tiết.

**Vì sao phải chấm lại:**
- Deformable 3DGS và SpectroMotion chỉ có PSNR/SSIM/LPIPS toàn ảnh, chưa có số cho vùng kính.
- Chúng render ở độ phân giải gốc (VSGD 1927×1080, Wayve 3271×1224), còn VIGOR-GS và RT-Splatting chấm ở `-r 2` (964×540 và 1636×612). Vì vậy các con số trong `results.json` của chúng không so trực tiếp được với VIGOR trên hai scene này.
- Thứ tự view test (`00000.png`, …) của các method không chắc giống nhau.

**Cách chấm** (mỗi method, mỗi scene):
1. Dựng ảnh tham chiếu từ `colmap/images` và `colmap/masks` của các frame test. Ảnh được resize tới kích thước của VIGOR-GS đúng như VIGOR-GS làm (`round(W/r)`, filter mặc định của PIL, mask > 0).
2. Ghép từng `gt/<i>.png` của method với một frame test, bằng cách so thumbnail (ghép 1–1). Sau đó so `gt/` của method với ảnh tham chiếu (cột `GT check PSNR`). **Dưới 35 dB nghĩa là method đó đã dùng ảnh khác**, ví dụ ảnh nguồn chưa undistort. Khi đó số của nó không so được, và notebook sẽ in cảnh báo.
3. Ảnh render khác kích thước được resize cùng filter với ảnh tham chiếu, rồi chấm.

**Các cột kết quả:**

| Cột | Ý nghĩa |
|---|---|
| `PSNR`, `SSIM`, `LPIPS` | Toàn ảnh, như `metrics.py` |
| `Masked PSNR/SSIM/LPIPS` | **Đúng định nghĩa của `metrics.py`:** nhân cả hai ảnh với mask rồi tính trên toàn khung. So được với bảng VIGOR/RT hiện có, nhưng bị thổi phồng vì các pixel ngoài mask đều bằng 0 ở cả hai ảnh |
| `PSNR in-mask` | PSNR thật trong vùng kính: MSE chỉ tính trên các pixel thuộc mask |
| `PSNR out-of-mask` | Tương tự, cho các pixel ngoài mask |
| `GT check PSNR` | So `gt/` của method với ảnh tham chiếu: kiểm tra method có dùng đúng dữ liệu không |

**Cần chuẩn bị** (các chỗ ⚠️ ở ô cấu hình):
1. Nhánh `d_masked_baseline_eval` đã được push lên GitHub (repo công khai `iamkyledang/vigor_gs`).
2. Zip của scene ở gốc My Drive, giống các notebook khác: `0758.zip`, `WayveScene101_scene_001.zip`, `3D-FRONT-T_scene_1.zip`. Notebook chỉ giải nén `colmap/frames.json`, `colmap/split.json`, cùng ảnh và mask của các frame test.
3. ⚠️ **Shortcut của thư mục chia sẻ `DACN_HK261` trong My Drive.** Kết quả của Deformable 3DGS và SpectroMotion nằm trong `DACN_HK261/results`, một thư mục được chia sẻ cho bạn. `drive.mount` chỉ thấy những gì có trong My Drive, nên cần tạo shortcut: vào drive.google.com → *Shared with me* → chuột phải `DACN_HK261` → *Organize* → *Add shortcut* → *My Drive*. Notebook tự tìm shortcut này, đặt ở thư mục nào trong My Drive cũng được. Nếu không tìm thấy, notebook vẫn chấm VIGOR/RT và báo bỏ qua hai method kia.
4. GPU không bắt buộc, nhưng LPIPS-VGG chạy trên GPU nhanh hơn nhiều.

**Truy cập Drive** (`ACCESS` ở ô cấu hình):

| `ACCESS` | Cách hoạt động | Khi nào dùng |
|---|---|---|
| `'mount'` (mặc định) | `drive.mount`, giống các notebook VIGOR | Cần shortcut `DACN_HK261` (mục 3 ở trên) |
| `'api'` | Drive API theo ID, qua `auth.authenticate_user()` | Không cần shortcut. Nhưng trên một số tài khoản, bước này lỗi `MessageError: credential propagation was unsuccessful`; khi đó dùng `'mount'` |

**Thứ tự và dọn dẹp:**
- Notebook chạy từng scene; trong mỗi scene, chạy từng method theo `ORDER`. **RT-Splatting đứng cuối**, vì nó còn đang được chạy lại trên Colab.
- Mỗi method xong thì xoá ảnh đã tải về. Mỗi scene xong thì xoá zip đã tải và dữ liệu đã giải nén, trừ khi `KEEP_DATA = True`.
- Kết quả được lưu ở hai chỗ:
  - Bản tổng hợp trong thư mục `masked_eval/` của `MyDrive/vigor_gs_runs`, gồm `<scene>/<method>.json` và `summary.csv`.
  - Một bản `masked_eval.json` đặt **ngay trong thư mục kết quả của method đó trên Drive**, nếu `UPLOAD_TO_METHOD_FOLDER = True`.
- Chạy lại notebook thì method nào đã chấm và ảnh render không đổi sẽ được bỏ qua. Ảnh render được nhận diện bằng số file và thời gian sửa mới nhất. Khi RT-Splatting có kết quả mới, chỉ cần chạy lại notebook: chỉ RT được chấm lại.

In [ ]:
# ⚠️ ĐIỀN / KIỂM TRA các giá trị trong ô này
REPO = 'iamkyledang/vigor_gs'          # repo công khai trên GitHub
BRANCH = 'd_masked_baseline_eval'      # ⚠️ nhánh phải đã được push
REPO_DIR = '/content/vigor_gs'
ACCESS = 'mount'                       # 'mount' (drive.mount, cần shortcut DACN_HK261 trong My Drive) hoặc 'api' (Drive API theo ID)
MYDRIVE = '/content/drive/MyDrive'
DACN_ROOT = None                       # đường dẫn tới DACN_HK261 sau khi mount; None = tự tìm shortcut trong My Drive

# -r mà VIGOR-GS được chấm cho từng scene (giống colab_vsgd_0758.ipynb và colab_scene_suite.ipynb)
# 'paths' dùng khi ACCESS = 'mount' ({DACN} = thư mục DACN_HK261); các ID dùng khi ACCESS = 'api'.
# vigor_root: thư mục RUN_ROOT của scene trong MyDrive/vigor_gs_runs (chứa full/, full_mlp/, rtsplatting/, ...)
# deformable3dgs / spectromotion: thư mục của scene trong DACN_HK261/results/<method>/output/... (chứa test/ours_40000/)
SCENES = {
    'VSGD_0758': {
        'zip_name': '0758.zip', 'zip_id': None,
        'paths': {'zip': '{MYDRIVE}/0758.zip', 'vigor_root': '{MYDRIVE}/vigor_gs_runs/VSGD_0758',
                  'deformable3dgs': '{DACN}/results/Deformable-3D-Gaussians/output/vsgd/0758',
                  'spectromotion': '{DACN}/results/SpectroMotion/output/vsgd/0758'},
        'resolution': 2,
        'vigor_root': '14141yBIsLwu_I09SnfoDlib8IbZ8IT7p',        # vigor_gs_runs/VSGD_0758
        'deformable3dgs': '1kZmQCBR5xIa-R6Gt5dqiC9ZHEnXeY4se',    # Deformable-3D-Gaussians/output/vsgd/0758
        'spectromotion': '1O9j56pSFlEeXzxc8PM3UvI3CwAhtgm7z',     # SpectroMotion/output/vsgd/0758
    },
    'WayveScene101_scene_001': {
        'zip_name': 'WayveScene101_scene_001.zip', 'zip_id': '1ZGHcZ0y39dUzkJJ-qlXkddsnngdI0n3t',
        'paths': {'zip': '{MYDRIVE}/WayveScene101_scene_001.zip', 'vigor_root': '{MYDRIVE}/vigor_gs_runs/WayveScene101_scene_001_dai',
                  'deformable3dgs': '{DACN}/results/Deformable-3D-Gaussians/output/wayvescene101/scene_001',
                  'spectromotion': '{DACN}/results/SpectroMotion/output/wayvescene101/scene_001'},
        'resolution': 2,
        'vigor_root': '1oZixWx9dYAub9P3-cgZx_brP91YGsL1c',        # vigor_gs_runs/WayveScene101_scene_001_dai
        'deformable3dgs': '1suKFbX2SvWy70ONoOqwUz-A5PVXJzZ3W',    # Deformable-3D-Gaussians/output/wayvescene101/scene_001
        'spectromotion': '1QB-ujRLZeYDZ7Aa6KYTlRVUZc4NAP3ln',     # SpectroMotion/output/.../scene_001
    },
    '3D-FRONT-T_scene_1': {
        'zip_name': '3D-FRONT-T_scene_1.zip', 'zip_id': '1vkLS8kWyoSM1-BdB8dAXgkG7HYNzTbVL',
        'paths': {'zip': '{MYDRIVE}/3D-FRONT-T_scene_1.zip', 'vigor_root': '{MYDRIVE}/vigor_gs_runs/3D-FRONT-T_scene_1_dai',
                  'deformable3dgs': '{DACN}/results/Deformable-3D-Gaussians/output/3d-front-t/scene_1',
                  'spectromotion': '{DACN}/results/SpectroMotion/output/3d-front-t/scene_1'},
        'resolution': 1,
        'vigor_root': '155NOudD1aOxp3sp6z6Rg527PuwTbxElv',        # vigor_gs_runs/3D-FRONT-T_scene_1_dai
        'deformable3dgs': '1mfzop9yK3Gd6B0JilQ6zIg7-rnm0ofSM',    # Deformable-3D-Gaussians/output/3d-front-t/scene_1
        'spectromotion': '1_92RHR7Ftl1mtzkQyK0fIMz5ExS_sLf8',     # SpectroMotion/output/3d-front-t/scene_1
    },
}
SCENE_ORDER = ['VSGD_0758', 'WayveScene101_scene_001', '3D-FRONT-T_scene_1']

# key -> (nhãn, nơi tìm thư mục chạy). 'vigor:<tên>' = thư mục con <tên> của vigor_root.
METHODS = {
    'deformable3dgs': ('Deformable 3DGS', 'deformable3dgs'),
    'spectromotion': ('SpectroMotion', 'spectromotion'),
    'vigor_full': ('VIGOR-GS đầy đủ', 'vigor:full'),
    'vigor_full_mlp': ('VIGOR-GS đầy đủ + MLP Fresnel', 'vigor:full_mlp'),
    'vigor_no_separation': ('VIGOR-GS chỉ Phase I', 'vigor:no_separation'),
    'rtsplatting': ('RT-Splatting', 'vigor:rtsplatting'),
}
ORDER = ['deformable3dgs', 'spectromotion', 'vigor_full', 'vigor_full_mlp', 'vigor_no_separation', 'rtsplatting']  # RT-Splatting cuối cùng
REFERENCE = 'vigor_full'               # các cột Δ so từng view với method này

VIGOR_RUNS_ID = '1EkQ1XmKncRE49pTm1DGeVyLsdSx4PsBR'  # MyDrive/vigor_gs_runs (ACCESS = 'api'): bản tổng hợp được ghi vào <đây>/masked_eval/
VIGOR_RUNS_PATH = '{MYDRIVE}/vigor_gs_runs'           # như trên, khi ACCESS = 'mount'
OUT_FOLDER = 'masked_eval'
LOCAL_OUT = '/content/masked_eval_out'  # bản trong runtime, đồng bộ với Drive sau mỗi method
UPLOAD_TO_METHOD_FOLDER = True         # đặt thêm masked_eval.json vào thư mục kết quả của từng method trên Drive
RESULT_NAME = 'masked_eval.json'
WORK = '/content/masked_eval_work'     # chỗ tải ảnh tạm thời, xoá sau mỗi method
KEEP_DATA = False                      # True: giữ dữ liệu scene đã giải nén sau khi chấm xong scene đó
FORCE = False                          # True: chấm lại mọi thứ, kể cả kết quả đã có

In [ ]:
import os
import subprocess

print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], capture_output=True, text=True).stdout or 'Không có GPU: LPIPS chạy trên CPU (chậm hơn).')

import glob

service = None
if ACCESS == 'mount':
    from google.colab import drive

    drive.mount('/content/drive')
    if DACN_ROOT is None:   # shortcut của thư mục chia sẻ, ở My Drive hoặc trong một thư mục con
        candidates = [f'{MYDRIVE}/DACN_HK261', *glob.glob('/content/drive/.shortcut-targets-by-id/*/DACN_HK261'),
                      *glob.glob(f'{MYDRIVE}/*/DACN_HK261'), *glob.glob(f'{MYDRIVE}/*/*/DACN_HK261')]
        DACN_ROOT = next((c for c in candidates if os.path.isdir(os.path.join(c, 'results'))), None)
    if DACN_ROOT is None:
        print('[ WARN ] Không thấy DACN_HK261 trong My Drive: Deformable 3DGS và SpectroMotion sẽ bị bỏ qua.\n'
              '         Tạo shortcut: drive.google.com -> Shared with me -> chuột phải DACN_HK261 -> Organize -> Add shortcut -> My Drive,\n'
              '         rồi chạy lại ô này (hoặc điền DACN_ROOT ở ô cấu hình).')
    else:
        print('DACN_HK261:', DACN_ROOT)
elif ACCESS == 'api':
    from google.colab import auth
    from googleapiclient.discovery import build

    try:
        auth.authenticate_user()
    except Exception as error:
        raise RuntimeError(
            f'Đăng nhập Drive API thất bại ({error}). Đặt ACCESS = "mount" ở ô cấu hình '
            '(cần shortcut DACN_HK261 trong My Drive) rồi chạy lại từ ô cấu hình.'
        ) from None
    service = build('drive', 'v3', cache_discovery=False)
    print('Drive API: đã đăng nhập bằng', service.about().get(fields='user(emailAddress)').execute()['user']['emailAddress'])
else:
    raise ValueError(f"ACCESS phải là 'mount' hoặc 'api', không phải {ACCESS!r}")
os.makedirs(LOCAL_OUT, exist_ok=True)

## Lấy code từ nhánh `d_masked_baseline_eval`

Chỉ cần phần Python thuần (`eval_masked_baselines.py`, `utils/`, `lpipsPyTorch/`), không build CUDA extension.

In [ ]:
import os
import subprocess
import sys


def _git(*args):
    result = subprocess.run(['git', *args], capture_output=True, text=True, env={**os.environ, 'GIT_TERMINAL_PROMPT': '0'})
    if result.returncode:
        raise RuntimeError(f'git {args[0]} thất bại:\n{result.stderr or result.stdout}')
    return result.stdout.strip()


url = f'https://github.com/{REPO}.git'
if os.path.isdir(os.path.join(REPO_DIR, '.git')):
    _git('-C', REPO_DIR, 'fetch', '-q', url, BRANCH)
    _git('-C', REPO_DIR, 'reset', '-q', '--hard', 'FETCH_HEAD')
else:
    _git('clone', '-q', '--depth', '1', '--branch', BRANCH, '--single-branch', url, REPO_DIR)
CODE_VERSION = _git('-C', REPO_DIR, 'log', '-1', '--format=%h %s (%ci)')
print('Code:', CODE_VERSION)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
import importlib

import torch

import eval_masked_baselines as emb

importlib.reload(emb)
from lpipsPyTorch import LPIPS

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
LPIPS_MODEL = LPIPS(net_type='vgg').to(DEVICE)   # cùng mạng với metrics.py

## Đọc và ghi Drive

Các hàm dưới đây dùng Drive API. Khi `ACCESS = "mount"`, ô ngay sau phần dữ liệu thay chúng bằng bản đọc/ghi file trên ổ đã mount; khi đó "id" của một thư mục chính là đường dẫn của nó.

In [ ]:
import io
import json
import os
import re

if ACCESS == 'api':
    from googleapiclient.http import MediaFileUpload, MediaIoBaseDownload

FOLDER = 'application/vnd.google-apps.folder'


def children(folder_id):
    items, token = [], None
    while True:
        response = service.files().list(
            q=f"'{folder_id}' in parents and trashed = false",
            fields='nextPageToken, files(id, name, mimeType, size, modifiedTime)',
            pageSize=1000, pageToken=token, supportsAllDrives=True, includeItemsFromAllDrives=True,
        ).execute()
        items += response.get('files', [])
        token = response.get('nextPageToken')
        if not token:
            return items


def child(folder_id, name, folder=None):
    hits = [c for c in children(folder_id) if c['name'] == name and (folder is None or (c['mimeType'] == FOLDER) == folder)]
    return hits[0] if hits else None


def test_location(run_id):
    """renders/ and gt/ of the latest test/<ours|direct>_<iter>/ of a run folder, or None."""
    test = child(run_id, 'test', folder=True)
    if test is None:
        return None
    methods = [c for c in children(test['id']) if c['mimeType'] == FOLDER and re.fullmatch(r'(ours|direct)_\d+', c['name'])]
    if not methods:
        return None
    method = max(methods, key=lambda c: int(c['name'].rsplit('_', 1)[1]))
    renders, gt = child(method['id'], 'renders', folder=True), child(method['id'], 'gt', folder=True)
    if renders is None or gt is None:
        return None
    return {'run_id': run_id, 'method_dir': method['name'], 'renders_id': renders['id'], 'gt_id': gt['id']}


def fingerprint(folder_id):
    files = [c for c in children(folder_id) if c['mimeType'] != FOLDER]
    return f"{len(files)} files, latest {max((c['modifiedTime'] for c in files), default='-')}"


def download_file(file_id, path):
    request = service.files().get_media(fileId=file_id, supportsAllDrives=True)
    with io.FileIO(path, 'wb') as f:
        downloader = MediaIoBaseDownload(f, request, chunksize=32 * 1024 * 1024)
        done = False
        while not done:
            _, done = downloader.next_chunk()


def download_folder(folder_id, dest):
    os.makedirs(dest, exist_ok=True)
    files = [c for c in children(folder_id) if c['mimeType'] != FOLDER]
    for c in files:
        path = os.path.join(dest, c['name'])
        if not (os.path.exists(path) and str(os.path.getsize(path)) == c.get('size')):
            download_file(c['id'], path)
    return len(files)


def read_json(folder_id, name):
    found = child(folder_id, name, folder=False)
    if found is None:
        return None
    buffer = io.BytesIO()
    downloader = MediaIoBaseDownload(buffer, service.files().get_media(fileId=found['id'], supportsAllDrives=True))
    done = False
    while not done:
        _, done = downloader.next_chunk()
    return json.loads(buffer.getvalue())


def upload(local_path, parent_id, name, mimetype='application/json'):
    """Create or replace <parent>/<name> on Drive."""
    media = MediaFileUpload(local_path, mimetype=mimetype, resumable=False)
    existing = child(parent_id, name, folder=False)
    if existing:
        service.files().update(fileId=existing['id'], media_body=media, supportsAllDrives=True).execute()
    else:
        service.files().create(body={'name': name, 'parents': [parent_id]}, media_body=media, fields='id', supportsAllDrives=True).execute()


def ensure_folder(parent_id, name):
    found = child(parent_id, name, folder=True)
    if found:
        return found['id']
    return service.files().create(body={'name': name, 'mimeType': FOLDER, 'parents': [parent_id]}, fields='id', supportsAllDrives=True).execute()['id']


def find_zip(name):
    """Drive id of the zip called `name` (following shortcuts); fails if there are none or several different ones."""
    q = f"name = '{name}' and trashed = false"
    hits = service.files().list(q=q, fields='files(id, name, mimeType, size, shortcutDetails)', pageSize=50,
                                supportsAllDrives=True, includeItemsFromAllDrives=True, corpora='allDrives').execute().get('files', [])
    ids = {h['shortcutDetails']['targetFileId'] if h['mimeType'] == 'application/vnd.google-apps.shortcut' else h['id'] for h in hits}
    if len(ids) != 1:
        raise RuntimeError(f'Tìm thấy {len(ids)} file tên {name} trên Drive: điền zip_id của scene ở ô cấu hình')
    return ids.pop()


def run_folder(scene, where):
    """Drive id of the run folder of one method on one scene, or None."""
    cfg = SCENES[scene]
    if where.startswith('vigor:'):
        found = child(cfg['vigor_root'], where.split(':', 1)[1], folder=True)
        return found['id'] if found else None
    return cfg.get(where)

## Dữ liệu scene: chỉ giải nén phần cần cho split test

In [ ]:
import json
import os
import shutil
import zipfile


def extract_test_data(zip_path, scene_dir):
    """colmap/frames.json, colmap/split.json and the images + masks of the test frames, from the scene zip."""
    if not os.path.exists(zip_path):
        raise FileNotFoundError(f'Không thấy {zip_path}: kiểm tra SCENES ở ô cấu hình')
    stamp = f'{os.path.getsize(zip_path)}-{int(os.path.getmtime(zip_path))}'
    marker = os.path.join(scene_dir, '.extracted_from_zip')
    if os.path.exists(marker) and open(marker).read() == stamp:
        return
    shutil.rmtree(scene_dir, ignore_errors=True)
    with zipfile.ZipFile(zip_path) as z:
        entries = {info.filename.replace('\\', '/'): info for info in z.infolist()}
        hits = [name for name in entries if name.endswith('colmap/frames.json')]
        if len(hits) != 1:
            raise RuntimeError(f'Cần đúng một colmap/frames.json trong {zip_path}, tìm thấy {len(hits)}')
        prefix = hits[0][: -len('colmap/frames.json')]

        def take(rel):
            info = entries.get(prefix + rel)
            if info is None:
                raise FileNotFoundError(f'{zip_path} không có {prefix + rel}')
            target = os.path.join(scene_dir, rel)
            os.makedirs(os.path.dirname(target), exist_ok=True)
            with z.open(info) as src, open(target, 'wb') as dst:
                shutil.copyfileobj(src, dst)

        take('colmap/frames.json')
        take('colmap/split.json')
        for name in emb.split_test_frames(scene_dir):
            take(f'colmap/images/{name}')
            take(f'colmap/masks/{os.path.splitext(name)[0]}.png')
    with open(marker, 'w') as f:
        f.write(stamp)


def fetch_zip(scene):
    """Download the scene zip from Drive (by zip_id, else found by name) into WORK/_zips; returns its local path."""
    cfg = SCENES[scene]
    file_id = cfg.get('zip_id') or find_zip(cfg['zip_name'])
    meta = service.files().get(fileId=file_id, fields='id, name, size, mimeType, shortcutDetails', supportsAllDrives=True).execute()
    if meta['mimeType'] == 'application/vnd.google-apps.shortcut':
        file_id = meta['shortcutDetails']['targetFileId']
        meta = service.files().get(fileId=file_id, fields='id, name, size', supportsAllDrives=True).execute()
    path = os.path.join(WORK, '_zips', meta['name'])
    os.makedirs(os.path.dirname(path), exist_ok=True)
    if not (os.path.exists(path) and str(os.path.getsize(path)) == meta.get('size')):
        print(f"  tải {meta['name']} ({int(meta.get('size', 0)) / 1e9:.2f} GB) ...")
        download_file(file_id, path)
    return path

## Chế độ `drive.mount`: đọc/ghi file trên ổ đã mount

Khi `ACCESS = "mount"`, ô này thay các hàm Drive API ở trên. "id" của một thư mục là đường dẫn của nó. Zip được đọc thẳng từ Drive, không chép và không bao giờ bị xoá.

In [ ]:
import json
import os
import shutil

if ACCESS == 'mount':
    def resolve(template):
        if '{DACN}' in template and DACN_ROOT is None:
            return None
        return template.format(MYDRIVE=MYDRIVE, DACN=DACN_ROOT)

    def children(path):
        items = []
        for entry in os.scandir(path):
            stat = entry.stat()
            items.append({'id': entry.path, 'name': entry.name, 'mimeType': FOLDER if entry.is_dir() else 'file',
                          'size': str(stat.st_size), 'modifiedTime': f'{stat.st_mtime:.0f}'})
        return items

    def download_file(source, target):
        shutil.copyfile(source, target)

    def read_json(folder, name):
        path = os.path.join(folder, name)
        if not os.path.isfile(path):
            return None
        with open(path) as f:
            return json.load(f)

    def upload(local_path, parent, name, mimetype=None):
        os.makedirs(parent, exist_ok=True)
        shutil.copyfile(local_path, os.path.join(parent, name))

    def ensure_folder(parent, name):
        path = os.path.join(parent, name)
        os.makedirs(path, exist_ok=True)
        return path

    def fetch_zip(scene):
        path = resolve(SCENES[scene]['paths']['zip'])
        if not os.path.isfile(path):
            raise FileNotFoundError(f'Không thấy {path}: kiểm tra paths/zip của {scene} ở ô cấu hình')
        return path   # đọc thẳng từ Drive

    def run_folder(scene, where):
        paths = SCENES[scene]['paths']
        if where.startswith('vigor:'):
            path = os.path.join(resolve(paths['vigor_root']), where.split(':', 1)[1])
        else:
            path = resolve(paths[where]) if where in paths else None
        return path if path and os.path.isdir(path) else None

    VIGOR_RUNS_ID = resolve(VIGOR_RUNS_PATH)
    print('Chế độ drive.mount; kết quả ghi vào', os.path.join(VIGOR_RUNS_ID, OUT_FOLDER))

## Chấm

Ô này chạy lần lượt từng scene, và trong mỗi scene chạy từng method theo `ORDER`. Với mỗi method:
1. Tải `renders/` và `gt/` vào `WORK`.
2. Chấm.
3. Ghi `vigor_gs_runs/masked_eval/<scene>/<method>.json` trên Drive, và đặt `masked_eval.json` vào thư mục của method.
4. Xoá ảnh đã tải.

Số mà method tự báo trong `results.json` được giữ trong mục `reported` của file kết quả, để đối chiếu. Với VIGOR-GS và RT-Splatting, `Masked PSNR` chấm lại phải gần bằng số đã báo, vì cùng dữ liệu, cùng độ phân giải và cùng định nghĩa.

In [ ]:
import json
import os
import shutil
import time

for scene in SCENE_ORDER:
    cfg = SCENES[scene]
    scene_dir = os.path.join(WORK, '_data', scene)
    print(f'\n######## {scene}  (-r {cfg["resolution"]})')
    out_folder = ensure_folder(ensure_folder(VIGOR_RUNS_ID, OUT_FOLDER), scene)   # vigor_gs_runs/masked_eval/<scene>
    os.makedirs(os.path.join(LOCAL_OUT, scene), exist_ok=True)
    pending = []
    for key in ORDER:
        label, where = METHODS[key]
        run_id = run_folder(scene, where)
        location = test_location(run_id) if run_id else None
        if location is None:
            print(f'  {label:32s} bỏ qua: không thấy thư mục chạy hoặc test/<ours_N>/renders+gt ({where})')
            continue
        stamp = fingerprint(location['renders_id'])
        previous = None if FORCE else read_json(out_folder, f'{key}.json')
        if previous and previous.get('source', {}).get('renders_fingerprint') == stamp:
            with open(os.path.join(LOCAL_OUT, scene, f'{key}.json'), 'w') as f:
                json.dump(previous, f, indent=1)
            print(f'  {label:32s} đã chấm, ảnh render không đổi: bỏ qua')
            continue
        pending.append((key, label, run_id, location, stamp))
    if not pending:
        continue
    extract_test_data(fetch_zip(scene), scene_dir)
    for key, label, run_id, location, stamp in pending:
        out_path = os.path.join(LOCAL_OUT, scene, f'{key}.json')
        start = time.time()
        work = os.path.join(WORK, scene, key)
        n = download_folder(location['renders_id'], os.path.join(work, 'renders'))
        download_folder(location['gt_id'], os.path.join(work, 'gt'))
        try:
            result = emb.evaluate(os.path.join(work, 'renders'), os.path.join(work, 'gt'), scene_dir, cfg['resolution'], label, LPIPS_MODEL, DEVICE)
        finally:
            shutil.rmtree(work, ignore_errors=True)   # xoá ảnh đã tải dù chấm lỗi
        reported = read_json(run_id, 'results.json')
        result['source'] = {**location, 'scene': scene, 'method_key': key, 'renders_fingerprint': stamp, 'code': CODE_VERSION}
        result['reported'] = reported.get(location['method_dir']) if reported else None
        with open(out_path, 'w') as f:
            json.dump(result, f, indent=1)
        upload(out_path, out_folder, f'{key}.json')   # bản tổng hợp trên Drive
        if UPLOAD_TO_METHOD_FOLDER:
            try:
                upload(out_path, run_id, RESULT_NAME)
            except Exception as error:  # không có quyền ghi: vẫn còn bản trong masked_eval/
                print(f'  [ WARN ] không ghi được {RESULT_NAME} vào thư mục của {label}: {error}')
        m = result['mean']
        print(f"  {label:32s} {location['method_dir']:12s} {n} view, render {result['render_sizes']} -> {result['eval_size']}  "
              f"PSNR {m['PSNR']:.2f}  Masked {m.get('Masked PSNR', float('nan')):.2f}  in-mask {m.get('PSNR in-mask', float('nan')):.2f}  "
              f"GT check {m['GT check PSNR']:.1f} dB  ({time.time() - start:.0f} s)")
        for warning in result['warnings']:
            print('    [ WARN ]', warning)
    if not KEEP_DATA:   # xoá zip đã tải và dữ liệu đã giải nén của scene
        for folder in (os.path.join(WORK, '_data'), os.path.join(WORK, '_zips'), os.path.join(WORK, scene)):
            shutil.rmtree(folder, ignore_errors=True)

## Bảng tổng hợp

Mỗi scene một bảng. `Δ … vs VIGOR` là chênh lệch trung bình theo từng frame test so với `REFERENCE`, ghép theo tên frame chứ không theo số thứ tự file. Cột `thắng` là số frame mà VIGOR-GS cao hơn. Cột `PSNR báo cáo` là số method tự ghi trong `results.json`; cột này chỉ so được với cột `PSNR` khi hai bên cùng độ phân giải.

Bảng được lưu thành `vigor_gs_runs/masked_eval/summary.csv` trên Drive. Ô này đọc kết quả từ Drive, nên chạy được cả trong một phiên Colab mới.

In [ ]:
import json
import os

import numpy as np
import pandas as pd

COLUMNS = ['PSNR', 'SSIM', 'LPIPS', 'Masked PSNR', 'Masked SSIM', 'Masked LPIPS', 'PSNR in-mask', 'PSNR out-of-mask', 'GT check PSNR']


def paired(result, reference, key):
    common = [n for n in reference['per_view'] if n in result['per_view'] and reference['per_view'][n].get(key) is not None and result['per_view'][n].get(key) is not None]
    if not common:
        return None, None
    d = np.array([reference['per_view'][n][key] - result['per_view'][n][key] for n in common])
    return d.mean(), f'{int((d > 0).sum())}/{len(d)}'


out_root = ensure_folder(VIGOR_RUNS_ID, OUT_FOLDER)
rows = []
for scene in SCENE_ORDER:
    scene_folder = child(out_root, scene, folder=True)
    results = {}
    for key in ORDER:
        result = read_json(scene_folder['id'], f'{key}.json') if scene_folder else None
        if result is not None:
            results[key] = result
    reference = results.get(REFERENCE)
    for key, result in results.items():
        row = {'scene': scene, 'method': METHODS[key][0], 'checkpoint': result['source']['method_dir'],
               'render size': 'x'.join(map(str, result['render_sizes'][0])), 'eval size': 'x'.join(map(str, result['eval_size'])),
               'views': result['n_views'], **{c: result['mean'].get(c) for c in COLUMNS},
               'PSNR báo cáo': (result.get('reported') or {}).get('PSNR'),
               'Masked PSNR báo cáo': (result.get('reported') or {}).get('Masked PSNR'),
               'cảnh báo': ' | '.join(result['warnings'])}
        if reference is not None and key != REFERENCE:
            for metric in ('PSNR', 'PSNR in-mask', 'Masked PSNR'):
                row[f'Δ {metric} (VIGOR − method)'], row[f'VIGOR thắng ({metric})'] = paired(result, reference, metric)
        rows.append(row)

summary = pd.DataFrame(rows)
summary_path = os.path.join(LOCAL_OUT, 'summary.csv')
summary.to_csv(summary_path, index=False)
upload(summary_path, out_root, 'summary.csv', mimetype='text/csv')
pd.set_option('display.float_format', lambda v: f'{v:.4f}')
for scene, table in summary.groupby('scene', sort=False):
    print(f'\n######## {scene}')
    display(table.drop(columns='scene').set_index('method'))
print('Đã lưu vigor_gs_runs/masked_eval/summary.csv trên Drive và', summary_path)

## Cách đọc kết quả

- **`GT check PSNR` < 35 dB:** method đó không được chấm trên ảnh `colmap/` hiện tại, ví dụ nó dùng ảnh nguồn chưa undistort. Khi đó hãy chạy lại method này trên dữ liệu `colmap/` thay vì dùng số của nó.
- **`PSNR in-mask`** là con số nên đưa vào paper cho vùng kính. **`Masked PSNR`** chỉ dùng để so với các bảng VIGOR/RT hiện có, vì nó cao hơn PSNR thật trong mask.
- **`PSNR out-of-mask`** cho biết khoảng chênh với Deformable 3DGS nằm ở vùng kính hay ở vùng ngoài kính.
- **Lưu ý về resize:** ảnh render ở độ phân giải gốc được thu nhỏ bằng cùng filter với ảnh tham chiếu. Ảnh thu nhỏ thì mịn hơn, nên method render ở độ phân giải cao có thể được lợi một chút so với method render trực tiếp ở `-r 2`.